# Audio scream / distress detector

Trains the audio modality for the Live Multimodal Monitoring System.

**Recipe (identical to the video models):** EfficientNet-B0 over a log-mel
spectrogram treated as an image. Same backbone, same ImageNet normalisation,
same checkpoint format — so `AudioModelAdapter` loads the result with no code
changes.

## Dataset

Folder layout the dataset cell expects:

```
data/audio/
  NORMAL/   *.wav    speech, traffic, music, TV, room tone
  SCREAM/   *.wav    screams / shouts / distress
```

### Sources

| Class | Dataset (Kaggle) | Notes |
|---|---|---|
| SCREAM + NORMAL | **Human Screaming Detection Dataset** | 862 scream + 2,631 non-scream *vocal* clips, 44.1 kHz, mostly ~10 s |
| NORMAL (mix in) | **environmental-sound-classification-50** (ESC-50) | 2,000 everyday sounds, **no scream class** — use it purely as negatives |

Two properties of the Human Screaming dataset that the dataset cell compensates for:

* **Its negatives are all *vocal*** (speech, laughter, other vocalisations).
  Great for scream-vs-speech, but the model never sees a slammed door, music, or
  traffic. Mixing ESC-50 into `NORMAL/` fixes that — the prep cell below does it.
* **Scream clips are ~10 s, a real scream is ~1–2 s.** Cut into 1 s windows,
  most windows from a SCREAM file are silence or buildup. The dataset cell drops
  low-energy SCREAM windows so they don't become mislabelled positives — this
  is the difference between decent and terrible **precision**.

### Playback robustness
You will demo by playing screams through a speaker. `augment()` simulates that
path (gain / EQ / noise). Recording a few played-back clips into `SCREAM/` helps
more.

In [ ]:
!pip -q install soundfile
import os, glob, math, random, json
import numpy as np, torch, torch.nn as nn
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
# ---------------------------------------------------------------------------
# Build data/audio/{NORMAL,SCREAM} from the Kaggle datasets you added as Inputs.
# First look at what you actually have:
#     !ls /kaggle/input
#     !find /kaggle/input -maxdepth 4 -type d
# then fix the globs below to match the real subfolder names.
# ---------------------------------------------------------------------------
import shutil

DATA_ROOT = "data/audio"
SOURCES = [
    ("/kaggle/input/human-screaming-detection-dataset/**/*[Ss]cream*/**/*.wav", "SCREAM"),
    ("/kaggle/input/human-screaming-detection-dataset/**/[Pp]ositive*/**/*.wav", "SCREAM"),
    ("/kaggle/input/human-screaming-detection-dataset/**/[Nn]egative*/**/*.wav", "NORMAL"),
    ("/kaggle/input/human-screaming-detection-dataset/**/[Nn]on*[Ss]cream*/**/*.wav", "NORMAL"),
    # non-vocal negatives so the model is not speech-only:
    ("/kaggle/input/environmental-sound-classification-50/**/*.wav", "NORMAL"),
]

for c in ("NORMAL", "SCREAM"):
    os.makedirs(f"{DATA_ROOT}/{c}", exist_ok=True)

n = {"NORMAL": 0, "SCREAM": 0}
for pattern, cls in SOURCES:
    for src in glob.glob(pattern, recursive=True):
        if not src.lower().endswith((".wav", ".flac", ".ogg")):
            continue
        dst = f"{DATA_ROOT}/{cls}/{n[cls]:06d}_{os.path.basename(src)}"
        if not os.path.exists(dst):
            try:
                os.symlink(src, dst)          # instant, no extra disk
            except OSError:
                shutil.copy(src, dst)
            n[cls] += 1
print("linked:", n)
assert n["SCREAM"] and n["NORMAL"], (
    "Nothing matched. Run  !find /kaggle/input -name '*.wav' | head  and fix the globs."
)

## 1. Feature front-end

**Copied verbatim from `src/cctv_ai/inference/audio/features.py`.** Training and
inference must compute features identically — a mismatch silently destroys the
trained weights. If you change one, change both.

In [ ]:
SAMPLE_RATE, N_FFT, HOP_LENGTH, N_MELS = 16000, 400, 160, 64
F_MIN, F_MAX, CLIP_SECONDS, IMG_SIZE = 20.0, 7600.0, 1.0, 224
_IMAGENET_MEAN, _IMAGENET_STD = (0.485,0.456,0.406), (0.229,0.224,0.225)

def hz_to_mel(hz): return 2595.0*np.log10(1.0+np.asarray(hz,dtype=np.float64)/700.0)
def mel_to_hz(m):  return 700.0*(10.0**(np.asarray(m,dtype=np.float64)/2595.0)-1.0)

def mel_filterbank(sample_rate=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS,
                   f_min=F_MIN, f_max=F_MAX):
    n_freqs = n_fft//2+1
    fft_freqs = np.linspace(0, sample_rate/2.0, n_freqs)
    mel_points = np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels+2)
    hz_points = mel_to_hz(mel_points)
    fb = np.zeros((n_mels, n_freqs), dtype=np.float32)
    for i in range(n_mels):
        left, centre, right = hz_points[i], hz_points[i+1], hz_points[i+2]
        if right <= left: continue
        rising  = (fft_freqs-left)/max(centre-left,1e-9)
        falling = (right-fft_freqs)/max(right-centre,1e-9)
        fb[i] = np.clip(np.minimum(rising,falling), 0.0, None)
    return fb

FB = torch.as_tensor(mel_filterbank())

def fix_length(w, seconds=CLIP_SECONDS, sr=SAMPLE_RATE):
    target = int(round(seconds*sr)); n = w.shape[0]
    if n == target: return w
    if n < target:
        pad = target-n; return np.pad(w,(pad//2,pad-pad//2))
    s = (n-target)//2; return w[s:s+target]

def log_mel(w):
    x = torch.as_tensor(np.ascontiguousarray(w), dtype=torch.float32)
    spec = torch.stft(x, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=N_FFT,
                      window=torch.hann_window(N_FFT), center=True,
                      pad_mode="reflect", normalized=False, onesided=True,
                      return_complex=True)
    power = spec.real.pow(2)+spec.imag.pow(2)
    return torch.log(FB @ power + 1e-6)

def spectrogram_to_batch(w):
    mel = log_mel(fix_length(w))
    lo, hi = mel.min(), mel.max()
    mel = (mel-lo)/(hi-lo) if (hi-lo) > 1e-6 else torch.zeros_like(mel)
    img = mel.unsqueeze(0).unsqueeze(0)
    img = torch.nn.functional.interpolate(img, size=(IMG_SIZE,IMG_SIZE),
                                          mode="bilinear", align_corners=False)
    img = img.repeat(1,3,1,1)
    mean = torch.tensor(_IMAGENET_MEAN).view(1,3,1,1)
    std  = torch.tensor(_IMAGENET_STD).view(1,3,1,1)
    return ((img-mean)/std)[0]        # [3,224,224]

## 2. Dataset

Long files are sliced into `CLIP_SECONDS` windows so one recording yields many
examples. Augmentation simulates the played-through-a-speaker path you will
actually demo with.

In [ ]:
LABEL_MAP = {"NORMAL": 0, "SCREAM": 1}
DATA_ROOT = "data/audio"
SCREAM_WINDOW_MIN_DBFS = -45.0   # drop near-silent windows cut from ~10 s scream clips

def load_wav(path):
    w, sr = sf.read(path, dtype="float32", always_2d=False)
    if w.ndim > 1: w = w.mean(axis=1)
    if sr != SAMPLE_RATE:                          # nearest-sample decimation
        idx = (np.arange(int(len(w)*SAMPLE_RATE/sr))*sr/SAMPLE_RATE).astype(int)
        w = w[np.clip(idx, 0, len(w)-1)]
    return w.astype(np.float32)

def slice_windows(w, seconds=CLIP_SECONDS, hop=0.5):
    n = int(seconds*SAMPLE_RATE); step = max(1, int(hop*SAMPLE_RATE))
    if len(w) <= n: return [fix_length(w)]
    return [w[i:i+n] for i in range(0, len(w)-n+1, step)]

def dbfs(x):
    return 20.0*np.log10(max(float(np.sqrt(np.mean(np.square(x, dtype=np.float64)))), 1e-9))

def augment(w, rng):
    """Simulate speaker playback + phone mic: gain, EQ tilt, noise, clipping."""
    w = w * rng.uniform(0.4, 1.4)
    if rng.random() < 0.5:
        a = rng.uniform(-0.4, 0.4)
        w = np.convolve(w, np.array([1.0, a], np.float32), mode="same")
    if rng.random() < 0.5:
        w = w + rng.normal(0, rng.uniform(0.001, 0.02), len(w)).astype(np.float32)
    return np.clip(w, -1.0, 1.0).astype(np.float32)

class AudioDS(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
        self.rng = np.random.default_rng(SEED)
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        w, y = self.items[i]
        if self.train: w = augment(w, self.rng)
        return spectrogram_to_batch(w), y

# Split by FILE, then slice. Windows from one recording must never land in both
# train and val — that leak inflates val accuracy and makes the number a lie.
files = []
for name, y in LABEL_MAP.items():
    fs = glob.glob(os.path.join(DATA_ROOT, name, "**", "*.wav"), recursive=True)
    print(f"{name:8s} {len(fs):5d} files")
    files += [(f, y) for f in fs]
assert files, f"No audio under {DATA_ROOT}/<CLASS>/*.wav — run the prep cell first."

random.shuffle(files)
cut = int(0.85*len(files))
train_files, val_files = files[:cut], files[cut:]

def build(file_list):
    out = []
    for path, y in file_list:
        for win in slice_windows(load_wav(path)):
            if y == LABEL_MAP["SCREAM"] and dbfs(win) < SCREAM_WINDOW_MIN_DBFS:
                continue                          # silent slice of a scream clip
            out.append((win, y))
    return out

train_items = build(train_files); random.shuffle(train_items)
val_items = build(val_files)

counts = {k: sum(1 for _, y in train_items if y == v) for k, v in LABEL_MAP.items()}
print("train windows:", len(train_items), counts)
print("val   windows:", len(val_items))

train_dl = DataLoader(AudioDS(train_items, True),  batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(AudioDS(val_items,   False), batch_size=32, shuffle=False, num_workers=2)

## 3. Model — same backbone as the video models

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

class AudioSpectrogramClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        w = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        backbone = efficientnet_b0(weights=w)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone, self.head = backbone, nn.Linear(in_features, num_classes)
    def forward(self, x): return self.head(self.backbone(x))

model = AudioSpectrogramClassifier(len(LABEL_MAP)).to(DEVICE)

# Class weights: scream clips are usually far rarer than negatives.
n = np.array([counts[k] for k in LABEL_MAP], dtype=np.float64)
weights = torch.tensor((n.sum()/(len(n)*np.maximum(n,1))), dtype=torch.float32, device=DEVICE)
print("class weights:", weights.tolist())

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

## 4. Train

In [ ]:
from tqdm.auto import tqdm

EPOCHS = 8
best = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train(); tot = 0; correct = 0; loss_sum = 0.0
    bar = tqdm(train_dl, desc=f"epoch {epoch}/{EPOCHS}", leave=False)
    for xb, yb in bar:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        out = model(xb); loss = criterion(out, yb)
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * yb.size(0); tot += yb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        bar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{correct/max(tot,1):.3f}")
    tr_acc = correct / max(tot, 1)

    model.eval(); vt = 0; vc = 0
    tp = fp = fn = 0
    with torch.inference_mode():
        for xb, yb in tqdm(val_dl, desc="  val", leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb).argmax(1)
            vc += (pred == yb).sum().item(); vt += yb.size(0)
            tp += ((pred == 1) & (yb == 1)).sum().item()
            fp += ((pred == 1) & (yb == 0)).sum().item()
            fn += ((pred == 0) & (yb == 1)).sum().item()
    va = vc / max(vt, 1)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    print(f"epoch {epoch}  loss {loss_sum/max(tot,1):.4f}  train {tr_acc:.3f}  "
          f"val {va:.3f}  scream P {prec:.3f} R {rec:.3f} F1 {f1:.3f}")

    # Save on F1, not accuracy: with a 1:3 class imbalance, accuracy rewards a
    # model that mostly predicts NORMAL.
    if f1 > best:
        best = f1
        torch.save({
            "model_name": "audio",
            "arch": "efficientnet_b0_logmel",
            "label_map": LABEL_MAP,
            "sample_rate": SAMPLE_RATE,
            "clip_seconds": CLIP_SECONDS,
            "n_mels": N_MELS, "n_fft": N_FFT, "hop_length": HOP_LENGTH,
            "img_size": IMG_SIZE,
            "val_acc": va, "val_f1": f1,
            "state_dict": model.state_dict(),
        }, "audio_best.pt")
        print(f"   saved audio_best.pt  (F1 {f1:.3f}, P {prec:.3f}, R {rec:.3f})")

## 5. Precision matters more than accuracy here

A false scream calls a phone. Watch **precision** on the SCREAM class, not
overall accuracy — with imbalanced data a model that never predicts SCREAM can
still score 95% accuracy while being useless.

## 6. Install

1. Download `audio_best.pt` from `/kaggle/working`
2. Copy it to `models/audio_best.pt`
3. Set in `.env`:

```env
AUDIO_MODEL_WEIGHTS_PATH=models/audio_best.pt
```

4. Restart. `/api/status` should show `audio: loaded`, and with
   `FUSION_ENABLED=true` audio can now corroborate video.